# 端侧现场交付状态自动说明助手

> Intel AI PC 创新应用征文 & OpenClaw Skill 挑战赛作品 Notebook

这个 Notebook 展示一个完整的端侧多模态工作流：现场人员拍摄交付照片，系统在本地完成图片理解、通知生成、语音播报和交付留档。

```text
交付现场照片
  -> OpenVINO Qwen3-VL 识别物品、位置、参照物
  -> 场景化生成收件人通知
  -> OpenVINO Qwen3-TTS 生成播报音频
  -> OpenVINO Qwen3-ASR 回读验证/预留语音追问
  -> Markdown / JSON 交付留档
```

项目面向企业园区、医院科室、工厂仓库等弱网、内网和隐私敏感场景。图片、通知和语音都可以在端侧处理，避免把交付现场照片上传到外部服务。


## 1. 安装依赖

依赖对齐官方 `modelscope-workshop` baseline。


In [ ]:
!git clone https://github.com/Liyixine/2026-04-30-OPEN
%cd 2026-04-30-OPEN

In [ ]:
%pip install --upgrade pip
%pip install -r requirements.txt --upgrade


## 2. 导入模块与读取样例

样例清单只提供必要上下文，不预置最终回复。最终通知、TTS 文本和追问回答都由模型和代码实时生成。


In [ ]:
from pathlib import Path
import json
import time

import IPython.display as ipd
from IPython.display import Image, Markdown, display

from src.delivery_generator import generate_delivery_result
from src.delivery_prompts import build_vlm_observation_prompt
from src.openvino_vlm_service import DEFAULT_VLM_MODEL_DIR, DEFAULT_VLM_MODEL_ID, OpenVINODeliveryVLM, download_vlm_model
from src.sample_loader import context_from_sample, load_manifest
from src.workshop_services import load_tts_model, transcribe_audio_subprocess

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

samples = load_manifest("samples/manifest.example.json")
print(f"已加载 {len(samples)} 个交付场景样例")
for sample in samples:
    print(f"- {sample['id']}: {sample['scene']} -> {sample['image_path']}")


## 3. 输入素材展示

三张图片分别对应企业、医院、工厂三类交付现场。请注意，这里只是输入素材展示，不是最终输出。


In [ ]:
for sample in samples:
    print("\n" + "=" * 80)
    print(f"样例：{sample['id']}")
    print(sample["image_description"])
    display(Image(filename=sample["image_path"], width=430))


## 4. 下载并加载 OpenVINO VLM

默认模型：`snake7gun/Qwen3-VL-4B-Instruct-int4-ov`。

本项目采用官方 Lab 1 的加载方式。为保证生成结果适合交付场景，后续每张图都会使用带业务上下文的 VLM prompt。


In [ ]:
DEVICE_VLM = "AUTO"

start = time.time()
model_dir = download_vlm_model(DEFAULT_VLM_MODEL_ID, DEFAULT_VLM_MODEL_DIR)
print("模型目录:", model_dir)

vlm_service = OpenVINODeliveryVLM(model_dir=model_dir, device=DEVICE_VLM)
print(f"OpenVINO VLM 加载完成，用时 {time.time() - start:.1f}s")


## 5. VLM 图片理解与交付说明生成

这里会把“目标物品、收件对象、补充备注”写入 VLM prompt，避免模型把文件架、绿植、登记牌等参照物误判为交付物品。

输出会分为四块：

1. VLM 结构化观察
2. 收件人通知
3. TTS 播报文本
4. 二次追问回答


In [ ]:
outputs = []

for sample in samples:
    print("\n" + "=" * 90)
    print(f"处理样例：{sample['id']}")
    display(Image(filename=sample["image_path"], width=430))

    context = context_from_sample(sample)
    followup_question = ""
    if sample.get("followup_cases"):
        followup_question = sample["followup_cases"][0].get("question", "")

    prompt = build_vlm_observation_prompt(context, sample.get("image_description", ""))
    observation = vlm_service.observe_delivery_image(sample["image_path"], question=prompt)
    result = generate_delivery_result(context, observation, followup_question)

    record = {
        "sample_id": sample["id"],
        "image_path": sample["image_path"],
        "context": context.__dict__,
        "observation": observation.__dict__,
        "result": result.to_dict(),
    }
    outputs.append(record)

    print("【VLM 结构化观察】")
    print(json.dumps(observation.__dict__, ensure_ascii=False, indent=2))
    print("\n【收件人通知】")
    print(result.recipient_message)
    print("\n【TTS 播报文本】")
    print(result.tts_text)
    if result.followup_answer:
        print("\n【二次追问】")
        print("问：", followup_question)
        print("答：", result.followup_answer)

    (OUTPUT_DIR / f"{sample['id']}_record.md").write_text(result.archive_markdown, encoding="utf-8")

(OUTPUT_DIR / "delivery_results.json").write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n结构化结果已保存到 outputs/delivery_results.json")


## 6. OpenVINO TTS：生成语音播报

TTS 使用官方 Lab 3 的 `Qwen3-TTS-CustomVoice-0.6B-fp16-ov`。

为了让播报自然，TTS 不直接朗读完整留档，而是使用前面生成的短播报稿。默认只对第一个企业园区样例生成音频，便于控制运行时间。


In [ ]:
RUN_TTS = True
DEVICE_AUDIO = "CPU"

tts_audio_path = OUTPUT_DIR / "office_001_tts.wav"

if RUN_TTS:
    start = time.time()
    tts_model = load_tts_model(device=DEVICE_AUDIO)
    print(f"TTS 模型加载完成，用时 {time.time() - start:.1f}s")

    text = outputs[0]["result"]["tts_text"]
    print("TTS 输入文本：")
    print(text)

    start = time.time()
    wavs, sr = tts_model.generate_custom_voice(
        text=text,
        speaker="vivian",
        language="Chinese",
        instruct="用自然、清晰、礼貌的语气播报现场交付通知，语速适中，不要像朗读清单。",
        non_streaming_mode=True,
        max_new_tokens=2048,
    )
    tts_time = time.time() - start

    if wavs is not None:
        import soundfile as sf
        sf.write(tts_audio_path, wavs[0], sr)
        duration = len(wavs[0]) / sr
        print(f"语音已生成：{tts_audio_path}")
        print(f"音频时长: {duration:.2f}s，推理耗时: {tts_time:.2f}s，RTF: {tts_time / duration:.3f}")
        ipd.display(ipd.Audio(wavs[0], rate=sr))


## 7. OpenVINO ASR：独立进程回读验证

Qwen3-TTS 和 Qwen3-ASR 官方源码当前依赖的 `transformers` 版本不同。为了避免在同一个 Python 进程中互相覆盖，本 Notebook 将 ASR 放到独立子进程里执行。

这个单元用于回读上一节生成的 TTS 音频，也可以扩展为“语音追问”入口。


In [ ]:
RUN_ASR = True

if RUN_ASR:
    audio_for_asr = tts_audio_path if tts_audio_path.exists() else None
    if audio_for_asr is None:
        print("未找到 TTS 音频，跳过 ASR 回读。")
    else:
        asr_output_path = OUTPUT_DIR / "office_001_asr.json"
        asr_result = transcribe_audio_subprocess(
            audio_path=audio_for_asr,
            output_path=asr_output_path,
            device=DEVICE_AUDIO,
        )
        print("ASR 回读结果：")
        print(json.dumps(asr_result, ensure_ascii=False, indent=2))


## 8. 交付记录展示

每个样例都会导出一份 Markdown 留档，可用于企业内网、医院交接记录或工厂工单记录。


In [ ]:
for path in sorted(OUTPUT_DIR.glob("*_record.md")):
    print("\n" + "-" * 80)
    print(path)
    display(Markdown(path.read_text(encoding="utf-8")))


## 9. 可选：启动 Gradio 演示

如果灵感流环境允许端口访问，可以启动一个简易 Web 演示界面。比赛提交时，Notebook 截图通常已经足够展示核心流程。


In [ ]:
# from src.gradio_service import build_demo
# demo = build_demo()
# demo.launch(server_name="0.0.0.0", server_port=7860, share=False)


## 10. 小结

本 Notebook 已形成完整展示闭环：

- 三类隐私敏感交付场景
- OpenVINO VLM 实时图片理解
- 场景化送达说明生成
- 更自然的 TTS 播报文本
- OpenVINO TTS 音频生成
- OpenVINO ASR 独立进程回读验证
- Markdown 与 JSON 留档

下一步可以把运行截图、音频结果、GitHub 仓库地址和 Copaw Skill 运行截图整理到研习社文章中。
